<a href="https://colab.research.google.com/github/tatyanalitvin/python_for_ds_tasks/blob/dev/HW_2_1_%D0%9F%D0%BE%D0%B1%D1%83%D0%B4%D0%BE%D0%B2%D0%B0_%D0%BB%D1%96%D0%BD%D1%96%D0%B9%D0%BD%D0%BE%D1%97_%D1%80%D0%B5%D0%B3%D1%80%D0%B5%D1%81%D1%96%D1%97_%D0%B7%D0%B0_%D0%BE%D0%B4%D0%BD%D0%BE%D1%8E_%D0%BE%D0%B7%D0%BD%D0%B0%D0%BA%D0%BE%D1%8E.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Importing modules and initial setup

In [1]:
import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

import sklearn
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Helpers

In [3]:
# Loss/Cost function
def rmse_custom(actual, predicted):
  return np.sqrt(
      np.mean(
          (actual - predicted) ** 2
          )
      )

#rmse = np.sqrt(mean_squared_error(y, predictions)) # sikit learn lib

def _chart(X, y, w, b, name =''):
  fig = px.scatter(
      x=X,
      y=y,
      title=f'{name}: Прогнозування витрат за віком',
      labels={'x': 'age', 'y': 'charges'}
  )

  # Додаємо лінію регресії
  x_line = np.array([X.min(), X.max()])
  y_line = b + w * x_line

  fig.add_trace(
      go.Scatter(
          x=x_line,
          y=y_line,
          mode='lines',
          name=f'Лінія регресії {name}',
          line=dict(color='red', width=3)
      )
  )

  fig.show()


В цьому домашньому завданні кожне завдання оцінюється по 10 балів.

 # **Завдання 1.**
 Після перегляду лекцій про поняття функії, вступ до лінійної алгебри і мат. формулювання лін. регресії знайдіть найкращу лінію для прогнозу `charges` за `age` **для некурців** (датафрейм `non_smoker_df`) з допомогою

1. Методу МНК (з використанням тільки `numpy`, без `scikit learn`)

2. Full-Batch градієнтного спуску з `numpy` . Протестуйте 3 різних learning rate і зробіть висновок, який є найкращим виходячи з практик для цього, наведених в лекції. Зверніть увагу, що на вхід треба набір даних дворозміний, для цього можливо треба буде трансформувати Ваші дані X в формат, як був в лекції "Математичне формулювання лінійної регресії". Також, градієнтний спуск в нашому випадку може розходитись з навчальним рейтом 0.1, бо цей рейт в цій задачі завеликий. Спробуйте нижчі рейти.
3. З `scikit-learn.LinearRegression`. Тут зверніть увагу, що вхід `X` має бути двовимірним масивом, тому нам потрібно передати dataframe, а не окрему колонку. Якщо у Вас X - колонка (а у Вас так мало б бути), то можна скористатись `X.to_frame()` щоб конвертувати колонку в датафрейм.

Для кожного методу
- знайдіть і виведіть коефіцієнти моделі
- обчисліть прогнози моделі і збережіть в окрему змінну
- порахуйте точність прогнозу RMSE  

Для градієнтного спуску виведіть графік помилки в залежності від ітерації.

А також побудуйте на одному графіку дані `age` проти `charges` в вигляді діаграми розсіювання і всі три лінії регресії, знайдені кожним з методів (для град. спуску оберіть варіант з тим learning rate, який виявився найкращим).

Зробіть висновки, чи відрізняються результати моделей?
Чи є знайдены параметри моделы близькими до ваших найкращих припущень?

In [4]:
path = '/content/drive/MyDrive/Colab Notebooks/ML for People/Data/medical-charges.csv'
medical_df = pd.read_csv(path)
non_smoker_df = medical_df[medical_df.smoker == 'no']
non_smoker_df

,age,sex,bmi,children,smoker,region,charges
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
5,31,female,25.740,0,no,southeast,3756.62160
...,...,...,...,...,...,...,...
1332,52,female,44.700,3,no,southwest,11411.68500
1333,50,male,30.970,3,no,northwest,10600.54830
1334,18,female,31.920,0,no,northeast,2205.98080
1335,18,female,36.850,0,no,southeast,1629.83350


In [5]:
X_ns = non_smoker_df['age']
y_ns = non_smoker_df['charges']

# Трансформуємо X в двовимірний форма -> додаємо стовпець одиниць (для intercept)
# [1, x] для кожного зразка
# X_ns_with_intercept = np.c_[np.ones((X_ns.shape[0], 1)), X_ns] # OR
X_ns_with_intercept = np.column_stack([np.ones(len(X_ns)), X_ns])
X_ns[:3], X_ns_with_intercept[:3]

(1    18
 2    28
 3    33
 Name: age, dtype: int64,
 array([[ 1., 18.],
        [ 1., 28.],
        [ 1., 33.]]))

### 1: Метод МНК матричне рішення (ordinary least squares)


In [6]:
# https://numpy.org/doc/2.2/reference/generated/numpy.linalg.inv.html
# https://numpy.org/doc/2.2/reference/generated/numpy.dot.html#numpy.dot
# Dot product can be written in different ways in numpy:
# print(np.dot(a, b))
# print(a.dot(b))
# print(a @ b)

def ols_custom(X, y):
  theta = np.linalg.inv(X.T @ X) @ X.T @ y
  return theta

In [7]:
b_ols, w_ols = ols_custom(X_ns_with_intercept, y_ns)
b_ols, w_ols

(np.float64(-2091.4205565650805), np.float64(267.2489128311998))

In [8]:
y_ns_pred_ols = w_ols * X_ns + b_ols
y_ns_pred_ols

,age
1,2719.059874
2,5391.549003
3,6727.793567
4,6460.544654
5,6193.295741
...,...
1332,11805.522911
1333,11271.025085
1334,2719.059874
1335,2719.059874


In [9]:
ols_rmse = rmse_custom(y_ns, y_ns_pred_ols)
ols_rmse

np.float64(4662.505766636395)

In [10]:
print(f"""OLS
--------------------------------------------------
Коефіцієнти моделі:
w (weight/slope):       {w_ols}
b (bias/intercept):     {b_ols}

y(x) = {w_ols} * x + {b_ols}

RMSE: {ols_rmse}""")

_chart(X=X_ns, y=y_ns, w=w_ols, b=b_ols, name='МНК')

OLS
--------------------------------------------------
Коефіцієнти моделі:
w (weight/slope):       267.2489128311998
b (bias/intercept):     -2091.4205565650805
      
y(x) = 267.2489128311998 * x + -2091.4205565650805
      
RMSE: 4662.505766636395


### 2: Градієнтний спуск (full batch gradient descent)


In [11]:
def full_batch_gradient_descent_component_based(
    X, y,
    w0 = 0.0, b0 = 0.0,
    learning_rate = 0.01,
    epochs = 100,
    tolerance = 1e-6  # early stopping tolerance
    ):

    w = float(w0)
    b = float(b0)
    m = len(y)
    history_errors = []

    for _ in range(epochs):
        y_hat = w * X + b
        error = y_hat - y

        # fradients
        dw = (2/m) * np.dot(error, X)
        db = (2/m) * np.sum(error)

        w = w - learning_rate * dw
        b = b - learning_rate * db

        history_errors.append(rmse_custom(y, y_hat))

        # early stopping check
        if len(history_errors) > 1 and abs(history_errors[-1] - history_errors[-2]) < tolerance:
            break

    return w, b, history_errors


In [12]:
learning_rates = [0.0005, 0.0004, 0.0002, 0.00001]
results_fbgd = {}

for lr in learning_rates:
    w_fbgd, b_fbgd, costs = full_batch_gradient_descent_component_based(
        X_ns, y_ns,
        learning_rate=lr, epochs=200000
        )

    y_pred_fbgd = w_fbgd * X_ns + b_fbgd
    err = rmse_custom(y_ns, y_pred_fbgd)

    results_fbgd[lr] = {
        "w": w_fbgd,
        "b": b_fbgd,
        "costs": costs,
        "err": err
    }
    print(f"""Learning Rate {lr} -> RMSE = {err}
              w={w_fbgd}, b={b_fbgd},

          """)

Learning Rate 0.0005 -> RMSE = 4662.510181051035
              w=266.81969210482265, b=-2072.3572859663027,

          
Learning Rate 0.0004 -> RMSE = 4662.511284700876
              w=266.7690274339664, b=-2070.1070816518823,

          
Learning Rate 0.0002 -> RMSE = 4662.516805296405
              w=266.57017437488406, b=-2061.275286163626,

          
Learning Rate 1e-05 -> RMSE = 4684.0366468426355
              w=237.23824121686818, b=-758.536288220586,

          


In [13]:
# візуалізація навчання
fig = go.Figure()

for lr in learning_rates:
    fig.add_trace(go.Scatter(
        y=results_fbgd[lr]["costs"],
        mode="lines",
        name=f"LR: {lr}"
    ))

fig.update_layout(
    title="Learning Curves (RMSE vs Iterations)",
    xaxis_title="Epochs",
    yaxis_title="RMSE"
)

fig.show()

In [14]:
# найкращий LR
best_lr = min(results_fbgd, key=lambda lr: results_fbgd[lr]['err'])
best_w_fbgd = results_fbgd[best_lr]['w']
best_b_fbgd = results_fbgd[best_lr]['b']

print(f""" Full Batch Gradient Descent ->
Best Learning Rate: {best_lr}
------------------------------------------
Коефіцієнти найкращої моделі:
w (weight/slope):       {best_w_fbgd}
b (bias/intercept):     {best_b_fbgd}

y(x) = {best_w_fbgd} * x + {best_b_fbgd}

RMSE: {results_fbgd[best_lr]['err']}""")

_chart(X=X_ns, y=y_ns, w=best_w_fbgd, b=best_b_fbgd, name ='FBGD')

 Full Batch Gradient Descent -> 
Best Learning Rate: 0.0005
------------------------------------------
Коефіцієнти найкращої моделі:
w (weight/slope):       266.81969210482265
b (bias/intercept):     -2072.3572859663027
      
y(x) = 266.81969210482265 * x + -2072.3572859663027
      
RMSE: 4662.510181051035


In [15]:
y_pred_fbgd = best_w_fbgd * X_ns + best_b_fbgd
y_pred_fbgd

,age
1,2730.397172
2,5398.594093
3,6732.692553
4,6465.872861
5,6199.053169
...,...
1332,11802.266703
1333,11268.627319
1334,2730.397172
1335,2730.397172


### 3: scikit-learn.LinearRegression




In [16]:
lin_reg = LinearRegression()
lin_reg

LinearRegression()

In [17]:
lin_reg.fit(X_ns.to_frame(), y_ns)

LinearRegression()

In [18]:
lin_reg.coef_, lin_reg.intercept_

(array([267.24891283]), np.float64(-2091.4205565650864))

In [19]:
y_pred_sklearn = lin_reg.predict(X_ns.to_frame())
y_pred_sklearn

array([2719.0598744 , 5391.54900271, 6727.79356686, ..., 2719.0598744 ,
       2719.0598744 , 3520.80661289])

In [20]:
rmse_sklearn = np.sqrt(mean_squared_error(y_ns, y_pred_sklearn))

print(f"""Scikit-learn
-------------------------------------------------------
Коефіцієнти моделі:
w (weight/slope):       {lin_reg.coef_[0]}
b (bias/intercept):     {lin_reg.intercept_}

y(x) = {lin_reg.coef_[0]} * x + {lin_reg.intercept_}

RMSE: {rmse_sklearn}""")

_chart(X_ns, y_ns, w=lin_reg.coef_[0], b=lin_reg.intercept_, name ='Scikit-learn NON-SMOKERS')

Scikit-learn 
-------------------------------------------------------
Коефіцієнти моделі:
w (weight/slope):       267.2489128311997
b (bias/intercept):     -2091.4205565650864
      
y(x) = 267.2489128311997 * x + -2091.4205565650864
      
RMSE: 4662.505766636395


#### Predictions for plotting

In [21]:

x_range = np.linspace(X_ns.min(), X_ns.max(), 100)
x_range_b = np.column_stack([np.ones(len(x_range)), x_range])

y_ols_line = x_range_b @ (b_ols, w_ols)
y_gd_line = x_range_b @ (best_b_fbgd, best_w_fbgd)
y_sklearn_line = lin_reg.intercept_ + lin_reg.coef_[0] * x_range

fig = px.scatter(non_smoker_df, x="age", y="charges", title="Порівняння моделей регресії")

fig.add_trace(go.Scatter(x=x_range, y=y_ols_line, mode="lines", name="OLS (Numpy)", line=dict(dash="solid")))
fig.add_trace(go.Scatter(x=x_range, y=y_gd_line, mode="lines", name=f"GD (LR={best_lr})", line=dict(dash="dash")))
fig.add_trace(go.Scatter(x=x_range, y=y_sklearn_line, mode="lines", name="Scikit-learn", line=dict(dash="dot")))

fig.show()

#### Висновки & Спостереження:

Результати всіх трьох методів майже однакові:
- Коефіцієнти (w, b) співпадають з великою точністю, бачимо це на графіку та при виведених результатах
- RMSE ~4662.5058 для всіх методів

Градієнтний спуск надзвичайно повільний у даному випадку. Щоб досягти точності довелося знизити крок досить сильно і додати дуже багато ітерацій.


# **Завдання 2.**
Навчіть модель лінійної регресії з допомогою sklearn оцінювати розмір медичних збори для **курців** за їх віком.
Виведіть
- точність моделі
-  коефіцієнти
-  візуалізуйте модель у вигляді лінії на графіку розсіювання `age` проти `charges`

і зробіть висновки, чи це хороша модель, чи ви б її використовували в компанії?

In [22]:
smoker_df = medical_df[medical_df.smoker == 'yes']
smoker_df

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
11,62,female,26.290,0,yes,southeast,27808.72510
14,27,male,42.130,0,yes,southeast,39611.75770
19,30,male,35.300,0,yes,southwest,36837.46700
23,34,female,31.920,1,yes,northeast,37701.87680
...,...,...,...,...,...,...,...
1313,19,female,34.700,2,yes,southwest,36397.57600
1314,30,female,23.655,3,yes,northwest,18765.87545
1321,62,male,26.695,0,yes,northeast,28101.33305
1323,42,female,40.370,2,yes,southeast,43896.37630


In [23]:
X_s = smoker_df['age']
y_s = smoker_df['charges']

In [24]:
lin_reg_s = LinearRegression()
lin_reg_s

LinearRegression()

In [25]:
lin_reg_s.fit(X_s.to_frame(), y_s)

LinearRegression()

In [26]:
lin_reg_s.coef_, lin_reg_s.intercept_

(array([305.23760211]), np.float64(20294.128126915966))

In [27]:
y_s_pred_sklearn = lin_reg_s.predict(X_s.to_frame())
y_s_pred_sklearn

array([26093.642567  , 39218.85945773, 28535.54338388, 29451.25619021,
       30672.20659865, 29756.49379232, 27009.35537333, 28840.78098599,
       30977.44420076, 38608.38425351, 31282.68180287, 34945.53302819,
       31282.68180287, 37997.90904929, 25788.40496489, 36471.72103874,
       26398.88016911, 28840.78098599, 28535.54338388, 27009.35537333,
       31587.91940498, 34029.82022186, 37692.67144718, 38303.1466514 ,
       39829.33466195, 37387.43384507, 31893.15700709, 38913.62185562,
       26398.88016911, 39524.09705984, 29146.0185881 , 33724.58261975,
       26093.642567  , 30061.73139443, 30672.20659865, 29451.25619021,
       34335.05782397, 33114.10741553, 34945.53302819, 25788.40496489,
       29451.25619021, 33114.10741553, 25788.40496489, 39524.09705984,
       31282.68180287, 28535.54338388, 30977.44420076, 26093.642567  ,
       33114.10741553, 32503.63221131, 26093.642567  , 27314.59297544,
       39524.09705984, 25788.40496489, 39524.09705984, 36776.95864085,
      

In [30]:
rmse_s_sklearn = np.sqrt(mean_squared_error(y_s, y_s_pred_sklearn))

print(f"""Scikit-learn SMOKERS
-------------------------------------------------------
Коефіцієнти моделі:
w (weight/slope):       {lin_reg_s.coef_[0]}
b (bias/intercept):     {lin_reg_s.intercept_}

y(x) = {lin_reg_s.coef_[0]} * x + {lin_reg_s.intercept_}

RMSE: {rmse_s_sklearn}""")

_chart(X_s, y_s, w=lin_reg_s.coef_[0], b=lin_reg_s.intercept_, name ='Scikit-learn SMOKERS')

Scikit-learn SMOKERS
-------------------------------------------------------
Коефіцієнти моделі:
w (weight/slope):       305.2376021098288
b (bias/intercept):     20294.128126915966

y(x) = 305.2376021098288 * x + 20294.128126915966

RMSE: 10711.00334810241


#### Висновок & Спостереження

- RMSE для курців ~10711, що значно більше ніж для некурців ~4662
- Графік розсіювання показує значний розкид точок навколо лінії регресії, виглядає ніби два кластери :)
- Я б шукала б точнішу модель, шляхом збільшення кількості фіч для передбачення, чи дивилася б в бік інших моделей (мб нелінійних?).